[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/acceptance_pitch.ipynb)

# Pitch-v1 acceptance gate (Phase 3.5b)
Runs the pipeline on the held-out multi-field set (bake-off clip + clips from
>=2 other games/fields, >=1 entirely unseen in training).
Gates (aggregate): anchor coverage >= 40%, combined coverage >= 65%,
held-out reproj error <= 0.05 on every clip. Reports per-clip + per-field,
split unseen-field vs unseen-time.

In [ ]:
!pip install -q "git+https://github.com/PatrickJReed/soccer-vision.git#subdirectory=packages/soccer-vision[roboflow]"
from pathlib import Path

# --- weights: prefer a local upload, else pull from Drive soccer-vision/ ---
WEIGHTS = Path("/content/pitch_v1.pt")
if not WEIGHTS.exists():
    from google.colab import drive
    drive.mount("/content/drive")
    # the retrain saves best.pt (e22) and last.pt (e42); last.pt scored higher
    # on POSE mAP for this run — prefer it, fall back to best.pt.
    for cand in ("soccer-vision/last.pt", "soccer-vision/best.pt",
                 "soccer-vision/pitch_v1.pt"):
        src = Path("/content/drive/MyDrive") / cand
        if src.exists():
            !cp "{src}" /content/pitch_v1.pt
            print("using weights:", cand); break
assert WEIGHTS.exists(), "upload pitch weights or place last.pt in Drive soccer-vision/"
MAX_GAP = 45

# Each clip: (path, field_id, split).
#   INTERIM (today): just the bake-off clip. It is a SEEN field (in training),
#   so coverage here is FLATTERED — but it answers 'does the model yield usable
#   anchors through the real pipeline at all?'. Baseline to beat: the broadcast
#   model gave ~16% anchor coverage on this clip.
#   FULL gate: add >=1 game the model NEVER trained on (unseen_field) — that is
#   the real verdict; swap the list below once you have a third game's video.
CLIPS = [
    (Path("/content/bakeoff_clip.mp4"), "bakeoff", "seen_field_interim"),
    # (Path("/content/heldout_gameC.mp4"), "C", "unseen_field"),  # <- the real test
]


In [ ]:
import cv2
from soccer_vision.tracking.roboflow import RoboflowBackend
from soccer_vision.pitch.landmarks import build_frame_homographies
from soccer_vision.pitch.propagation import (
    compute_interframe_homographies, propagate_homographies,
)

backend = RoboflowBackend(detect_pitch=True, pitch_weights_path=WEIGHTS)
rows = []
for path, field, split in CLIPS:
    traj, kps = backend.process_with_pitch(path)
    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    def read_frame(i, _cap=cap):
        _cap.set(cv2.CAP_PROP_POS_FRAMES, i); ok, f = _cap.read(); return f if ok else None
    anchors = build_frame_homographies(kps)
    keys = sorted(anchors)
    needed = {i for a, b in zip(keys, keys[1:]) if b - a - 1 <= MAX_GAP for i in range(a, b)}
    interframe = compute_interframe_homographies(read_frame, needed, traj)
    homs = propagate_homographies(anchors, interframe, max_gap=MAX_GAP)
    cap.release()
    anchor_cov = len(anchors) / n
    combined_cov = len(homs) / n
    rows.append(dict(field=field, split=split, frames=n,
                     anchor_cov=anchor_cov, combined_cov=combined_cov))
    print(f"{field:8s} [{split}] anchor={anchor_cov:.1%} combined={combined_cov:.1%}")

In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
agg_anchor = df["anchor_cov"].mean()
agg_combined = df["combined_cov"].mean()
print(f"AGG anchor={agg_anchor:.1%} (gate >=40%): {'PASS' if agg_anchor>=0.40 else 'FAIL'}")
print(f"AGG combined={agg_combined:.1%} (gate >=65%): {'PASS' if agg_combined>=0.65 else 'FAIL'}")
print("baseline: broadcast pitch model gave ~16% anchor coverage on the bake-off clip\n")
if (df['split'] == 'unseen_field').any():
    print("By split:")
    print(df.groupby('split')[['anchor_cov','combined_cov']].mean())
else:
    print("INTERIM run (seen field only) — coverage is flattered; add an unseen "
          "game for the real gate.")
print("\nWorst field (anchor):", df.loc[df['anchor_cov'].idxmin(), 'field'])


Reprojection error and keypoint accuracy reuse the 3.5a held-out probe
(examples/colab_homography_propagation.ipynb) and a small labeled slice per
clip. Report median per-keypoint pixel error + per-landmark detection rate,
split unseen-field vs unseen-time. Reproj error must stay <= 0.05 on every clip.